In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置项
# ==========================================
# 独立 C 模型的文件路径
C_MODEL_PATH = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"

# 包含全局 0-100% 的 8 个经典切片 (刚好 2x4 拼图)
SLICES = [
    ("0-100% (Global)", 0.0, 1.00), 
    ("0-5%", 0.0, 0.05), 
    ("0-10%", 0.0, 0.10), 
    ("0-20%", 0.0, 0.20),
    ("10-20%", 0.10, 0.20), 
    ("20-50%", 0.20, 0.50), 
    ("50-100%", 0.50, 1.00), 
    ("20-100%", 0.20, 1.00)
]

# ==========================================
# 2. 核心算子：计算 AUUC/Qini 曲线
# ==========================================
def get_auuc_curve(target_s, t_s):
    if len(target_s) < 2: return [], [], 0.0
    n_t_cum = np.cumsum(t_s == 1)
    n_c_cum = np.cumsum(t_s == 0)
    
    y_t_cum = np.cumsum(target_s * (t_s == 1))
    y_c_cum = np.cumsum(target_s * (t_s == 0))

    n_t_safe = np.where(n_t_cum == 0, 1e-6, n_t_cum)
    n_c_safe = np.where(n_c_cum == 0, 1e-6, n_c_cum)
    x_axis = np.arange(1, len(target_s) + 1) / len(target_s)

    with np.errstate(invalid='ignore', divide='ignore'):
        uplift_curve = (y_t_cum / n_t_safe - y_c_cum / n_c_safe) * (n_t_cum + n_c_cum)
    uplift_curve = np.nan_to_num(uplift_curve)
    
    # 绝对值归一化
    endpoint = uplift_curve[-1]
    if abs(endpoint) > 1e-10:
        uplift_curve = uplift_curve / abs(endpoint) 
        
    return x_axis, uplift_curve, float(np.trapz(uplift_curve, x=x_axis))

# ==========================================
# 3. 核心绘图引擎
# ==========================================
def plot_8_grid(df, target_col, title, filename, line_color):
    """
    df: 必须已经按 Pred C 全局降序排好
    target_col: 'c_true' (检验导师能力) 或 'y_true' (检验跨界指导能力)
    """
    target_true = df[target_col].values 
    t = df['t'].values
    n_total = len(df)
    
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.suptitle(title, fontsize=22, fontweight='bold', y=0.98)
    
    for idx, (slice_name, start_pct, end_pct) in enumerate(SLICES):
        row = idx // 4
        col = idx % 4
        ax = axes[row, col]
        
        # 物理切片
        start_idx = int(n_total * start_pct)
        end_idx = int(n_total * end_pct)
        target_s = target_true[start_idx:end_idx]
        t_s = t[start_idx:end_idx]
        
        # 计算曲线
        x, curve, auuc_score = get_auuc_curve(target_s, t_s)
        
        if len(x) == 0: continue
            
        # 绘制曲线
        ax.plot(x, curve, color=line_color, linewidth=2.5, label='Uplift Curve')
        ax.fill_between(x, 0, curve, color=line_color, alpha=0.1)
        
        # 绘制随机基线
        end_y = curve[-1]
        ax.plot([0, 1], [0, end_y], linestyle='--', color='gray', label='Random (0.5)')
        
        # 标题与排版
        prefix = "C-AUUC" if target_col == 'c_true' else "Y-AUUC"
        ax.set_title(f"[{slice_name}]\n{prefix} = {auuc_score:.4f}", fontsize=15, fontweight='bold')
        ax.set_xlabel("Ratio in Slice", fontsize=12)
        ax.set_ylabel("Normalized Gain", fontsize=12)
        ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
        
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 生成完毕: {filename}")

# ==========================================
# 4. 执行流水线
# ==========================================
sns.set_theme(style="whitegrid")
output_dir = "0414_analyze_anatomy_c_model"
os.makedirs(output_dir, exist_ok=True)

print("🔍 正在加载独立 C 模型数据，启动导师资格审查...")

if not os.path.exists(C_MODEL_PATH):
    print(f"⚠️ 找不到文件: {C_MODEL_PATH}")
else:
    df = pd.read_csv(C_MODEL_PATH)
    
    # 🌟 核心：强行用 C 模型预测的增益进行全局降序！
    # （因为你存的时候没改名，它的 uplift_pred 实际上就是 C 的预测增益）
    df = df.sort_values(by='uplift_pred', ascending=False).reset_index(drop=True)
    
    # 任务一：画【C 排 C】(检验自身能力)
    title_c = "Prior Anatomy: How well does C-Model predict True Click (C)?\n(Sorted by Pred C)"
    file_c = os.path.join(output_dir, "prior_anatomy_C_predicts_C.png")
    plot_8_grid(df, 'c_true', title_c, file_c, line_color='teal')
    
    # 任务二：画【C 排 Y】(检验代理指导能力)
    title_y = "Proxy Anatomy: How well does Pred Click (C) rank True Conversion (Y)?\n(Sorted by Pred C)"
    file_y = os.path.join(output_dir, "prior_anatomy_C_predicts_Y.png")
    plot_8_grid(df, 'y_true', title_y, file_y, line_color='crimson')

print("\n🎉 全部审查完毕！")
print("💡 破案指南：")
print("1. 打开 C_predicts_C.png，如果 0-100% 和 0-5% 极其丝滑，说明导师能力很强。")
print("2. 打开 C_predicts_Y.png，如果 0-5% 曲线塌陷或者变成负数，说明用 C 的序直接去排 Y 是灾难，这也正是 V1 Baseline 会翻车、以及为什么我们需要 V8 解耦的核心铁证！")

🔍 正在加载独立 C 模型数据，启动导师资格审查...
✅ 生成完毕: 0414_analyze_anatomy_c_model/prior_anatomy_C_predicts_C.png
✅ 生成完毕: 0414_analyze_anatomy_c_model/prior_anatomy_C_predicts_Y.png

🎉 全部审查完毕！
💡 破案指南：
1. 打开 C_predicts_C.png，如果 0-100% 和 0-5% 极其丝滑，说明导师能力很强。
2. 打开 C_predicts_Y.png，如果 0-5% 曲线塌陷或者变成负数，说明用 C 的序直接去排 Y 是灾难，这也正是 V1 Baseline 会翻车、以及为什么我们需要 V8 解耦的核心铁证！
